**THỐNG KÊ SỐ LƯỢNG ẢNH TRONG TỪNG PHÂN LOẠI MÀ MỖI SINH VIÊN ĐÓNG GÓP**

In [1]:
import os
import csv
from collections import defaultdict

In [2]:
def process_directory(parent_dir):
    # Khởi tạo các biến để lưu trữ dữ liệu 
    student_image_count = defaultdict(float)
    student_brand_count = defaultdict(lambda: defaultdict(float))

    # Duyệt qua từng thư mục con (hãng xe)
    for brand in os.listdir(parent_dir):
        brand_path = os.path.join(parent_dir, brand)

        # Bỏ qua nếu không phải là thư mục hay không phải là thư mục hãng xe hợp lệ
        if not os.path.isdir(brand_path) or not brand.isalpha():
            continue

        # Duyệt qua từng hình ảnh trong thư mục
        for image_name in os.listdir(brand_path):
            if not image_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue

            # Tách thông tin từ tên tệp
            parts = image_name.split('.')
            if len(parts) < 3:
                continue

            #Xử lý thông tin mssv và hãng xe
            students_part = parts[0]
            brand_name = parts[1]

            # Tách mssv và phân bổ đóng góp mỗi sv
            student_ids = students_part.split('-') 
            num_students = len(student_ids)

            if num_students > 0:
                contribution = 1.0 / num_students

                # Cập nhật dữ liệu vào từ điển
                for student_id in student_ids:  # Với mỗi sv trong dsach
                    student_image_count[student_id] += contribution # Tăng tổng số lượng hình ảnh của sinh viên
                    student_brand_count[student_id][brand_name] += contribution # Tăng số lượng hình ảnh mà sv đóng góp cho nhãn xe cụ thể

    # Ghi dữ liệu số lượng hình ảnh mỗi sv đóng góp vào csv
    with open('CarDataset-1.csv', mode='w', newline='', encoding='utf-8') as file1:
        writer1 = csv.writer(file1)
        writer1.writerow(["MSSV", " All", " Số lượng"])
        last_student = None
        for student_id, count in sorted(student_image_count.items()):
            if last_student and student_id != last_student:
                writer1.writerow([]) 
            writer1.writerow([student_id, "All", round(count, 2)])
            last_student = student_id

    # Ghi dữ liệu số lượng hình ảnh mỗi sv đóng góp cho nhãn xe cụ thể vào csv
    with open('CarDataset-2.csv', mode='w', newline='', encoding='utf-8') as file2:
        writer2 = csv.writer(file2)
        writer2.writerow(["MSSV", " Hãng xe", " Số lượng"])
        last_student = None
        for student_id, brands in sorted(student_brand_count.items()):
            if last_student and student_id != last_student:
                writer2.writerow([]) 
            for brand_name, count in sorted(brands.items()):
                writer2.writerow([student_id, brand_name, round(count, 2)])
            last_student = student_id

    print("Kết quả đã được xuất vào CarDataset-1.csv và CarDataset-2.csv.")

In [3]:
parent_directory = "D:\\Dataset\\FinalData_CS114"
process_directory(parent_directory)

Kết quả đã được xuất vào CarDataset-1.csv và CarDataset-2.csv.


In [1]:
import pandas as pd

# Đường dẫn tới các file CSV
file1_path = 'D:\\VSCode\\CS114\\CarDataset-1.csv'
file2_path = 'D:\\VSCode\\CS114\\CarDataset-2.csv'

# Đọc dữ liệu từ file CSV
car_dataset_1 = pd.read_csv(file1_path)
car_dataset_2 = pd.read_csv(file2_path)

# Thống kê tổng số lượng đóng góp của mỗi sinh viên (CarDataset-1.csv)
summary_1 = car_dataset_1[' Số lượng'].describe()

print("Thống kê tổng số lượng đóng góp của mỗi sinh viên (CarDataset-1.csv):")
print(summary_1)

# Thống kê số lượng đóng góp theo từng hãng xe của mỗi sinh viên (CarDataset-2.csv)
summary_2 = car_dataset_2[' Số lượng'].describe()

print("\nThống kê số lượng đóng góp theo từng hãng xe của mỗi sinh viên (CarDataset-2.csv):")
print(summary_2)

# Tổng hợp số lượng đóng góp từ CarDataset-2.csv để so sánh với CarDataset-1.csv
total_contributions_by_student = car_dataset_2.groupby('MSSV')[' Số lượng'].sum()

# So sánh tổng hợp từ CarDataset-2.csv với CarDataset-1.csv
comparison = pd.merge(
    car_dataset_1[['MSSV', ' Số lượng']].rename(columns={' Số lượng': 'Overall Contribution'}),
    total_contributions_by_student.rename('Aggregated Contribution'),
    left_on='MSSV',
    right_index=True,
    how='left'
)

comparison['Difference'] = comparison['Overall Contribution'] - comparison['Aggregated Contribution']

# Hiển thị sự khác biệt (nếu có)
print("\nSo sánh tổng số lượng đóng góp giữa hai file:")
print(comparison)

# In ra các dòng có sự khác biệt (nếu có)
discrepancies = comparison[comparison['Difference'] != 0]
if not discrepancies.empty:
    print("\nCác khác biệt được tìm thấy:")
    print(discrepancies)
else:
    print("\nKhông có khác biệt giữa hai file.")


Thống kê tổng số lượng đóng góp của mỗi sinh viên (CarDataset-1.csv):
count      55.000000
mean      691.109636
std       924.567067
min         0.500000
25%       114.670000
50%       350.500000
75%       999.500000
max      3989.670000
Name:  Số lượng, dtype: float64

Thống kê số lượng đóng góp theo từng hãng xe của mỗi sinh viên (CarDataset-2.csv):
count    529.000000
mean      71.854442
std      126.824293
min        0.500000
25%        8.500000
50%       25.000000
75%       77.500000
max      800.000000
Name:  Số lượng, dtype: float64

So sánh tổng số lượng đóng góp giữa hai file:
         MSSV  Overall Contribution  Aggregated Contribution  Difference
0    20520918                240.00                   240.00        0.00
1    21520930                348.50                   348.50        0.00
2    21520938                523.00                   523.00        0.00
3    21522373                350.00                   350.00        0.00
4   215223773                  0.50       